# Experiment: First Quote Outbound Packet Assembly Gate

Objective:
- Classify whether the first quote packet is review-ready, send-blocked, held, or rejected.
- Keep founder approval and recipient qualification as hard gates.
- Prevent private data, patient samples, treatment, trial, purchase, import, or procurement scope.

In [ ]:
from __future__ import annotations

FOUNDER_REVIEW = "packet_ready_for_founder_review"
SEND_BLOCKED = "packet_send_blocked_no_approval"
HOLD = "packet_hold_missing_component"
REJECT = "packet_reject_private_or_clinical_scope"

REQUIRED_COMPONENTS = [
    "recipient_qualification_label",
    "public_safe_cover_note",
    "quote_table",
    "held_addenda_statement",
    "capability_questions",
    "public_source_links",
    "blocked_content_statement",
    "response_capture_path",
]

REJECT_FIELDS = [
    "raw_records",
    "patient_identifiers",
    "patient_samples",
    "doctor_messages",
    "treatment_or_dosing",
    "trial_screening",
    "travel",
    "purchase",
    "import",
    "procurement",
]

In [ ]:
def classify_packet(packet: dict[str, bool]) -> dict[str, object]:
    """Classify a synthetic outbound packet before any lab contact."""
    reject_fields = [field for field in REJECT_FIELDS if packet.get(field)]
    if reject_fields:
        return {"decision": REJECT, "reason": "blocked_scope", "fields": reject_fields}

    missing = [field for field in REQUIRED_COMPONENTS if not packet.get(field)]
    if missing:
        return {"decision": HOLD, "reason": "missing_component", "fields": missing}

    if not packet.get("founder_approval"):
        return {"decision": SEND_BLOCKED, "reason": "approval_missing", "fields": []}

    return {"decision": FOUNDER_REVIEW, "reason": "review_packet_ready", "fields": []}

## Synthetic Packet Cases

These examples model packet labels only. They do not name a recipient or send anything.

In [ ]:
complete_packet = {field: True for field in REQUIRED_COMPONENTS}
complete_packet.update({field: False for field in REJECT_FIELDS})
complete_packet["founder_approval"] = True

cases = {
    "no_approval": complete_packet | {"founder_approval": False},
    "review_ready": complete_packet,
    "missing_recipient": complete_packet | {"recipient_qualification_label": False},
    "private_data": complete_packet | {"raw_records": True},
    "procurement_scope": complete_packet | {"procurement": True},
}

results = {name: classify_packet(packet) for name, packet in cases.items()}
results

In [ ]:
expected = {
    "no_approval": SEND_BLOCKED,
    "review_ready": FOUNDER_REVIEW,
    "missing_recipient": HOLD,
    "private_data": REJECT,
    "procurement_scope": REJECT,
}

decision_summary = {name: result["decision"] for name, result in results.items()}
assert decision_summary == expected, decision_summary
decision_summary

## Decision

- A packet can be assembled for founder review.
- A packet cannot be sent without founder approval and recipient qualification.
- Any private, patient-specific, trial, purchase, import, or procurement scope rejects the packet.